In [ ]:
import sys
BASE_DIR = "/trinity/home/team06/workspace/mikhail_workspace/rag_project"
sys.path.insert(0, BASE_DIR)

from tqdm import tqdm
from typing import Dict, List
import numpy as np

from src.Reader import LLM_Model
from src.Scorer import SimilarityScorerConfig
from src.utils import ReaderMetrics, save_reader_trial_log, prepare_reader_configs, load_benchmarks_df
from src.utils import evaluate_reader

In [6]:
# !!! TO CHANGE !!!
TRIAL = 2
BENCHMARKS_MAXSIZE = 500
BENCHMARKS_INFO = {'mtssquad': {'table': 'v1'}}

READER_PARAMS = {
    'prompts': {
        "assistant": 'Отвечай на вопросы, используя информацию из текстов в списке ниже. Если ты не уверена в релевантности данных текстов по отношению к заданному вопросу, то сгенерируй следующий ответ: "У меня нет ответа на ваш вопрос.".',
        "system": "Ты вопросно-ответная система. Все ответы генерируй на русском языке. По вопросам отвечай кратко, чётко и конкретно. Не генерируй излишнюю информацию.",
    },
    'gen': {'max_new_tokens': 512, 'eos_token_id': 79097},
    'data_operate': {'batch_size': 5}
    }


BERTSCORE_MODEL_PATH = "ru_electra_medium"
# !!! TO CHANGE !!!

SAVE_LOGDIR = f'./logs/trial{TRIAL}'
SAVE_HYPERPARAMS = f'{SAVE_LOGDIR}/hyperparams.json'
SAVE_READERCACHE = f'{SAVE_LOGDIR}/reader_cache.json'

In [7]:
banchmarks_path = {}
for name, version in BENCHMARKS_INFO.items():
    banchmarks_path[name] = {
        'table': f"{BASE_DIR}/data/{name}/tables/{version['table']}/benchmark.csv",
    }

In [8]:
benchmarks_df = load_benchmarks_df(banchmarks_path, BENCHMARKS_MAXSIZE)

In [9]:
reader_config = prepare_reader_configs(READER_PARAMS)
READER = LLM_Model(reader_config)

Loading checkpoint shards: 100%|██████████| 4/4 [03:34<00:00, 53.62s/it]


In [10]:
sim_score_config = SimilarityScorerConfig()
reader_metrics = ReaderMetrics(BASE_DIR, BERTSCORE_MODEL_PATH, sim_score_config, READER)

Loading Meteor...
Loading ExactMatch


In [ ]:
reader_scores, reader_cache = evaluate_reader(benchmarks_df, READER, reader_metrics)

In [14]:
save_reader_trial_log(SAVE_LOGDIR, reader_scores, SAVE_HYPERPARAMS, SAVE_READERCACHE, 
                      reader_cache, BENCHMARKS_INFO, BENCHMARKS_MAXSIZE, READER_PARAMS)

In [15]:
reader_scores

{'mtssquad': {'BLEU2': 0.1908,
  'BLEU1': 0.26754,
  'ExactMatch': 0.002,
  'METEOR': 0.37203,
  'BertScore': {'precision': 0.36895,
   'recall': 0.37345,
   'f1': 0.37055,
   'hash': '/trinity/home/team06/workspace/mikhail_workspace/rag_project/models/ru_electra_medium_LNone_no-idf'},
  'StubScore': nan,
  'elapsed_time_sec': 487.471}}